### Testing RAG Applications - (Advanced ⚡️) 📑

#### RAG Application
This application reads data about Model Context Protocol (MCP) server from internet, stores in vector stores, chunks the data with embedding and useful to answer the question about MCP while inferenced.

<img src="./img/RAG.png" width="500" height="400" style="display: block; margin: auto;">

In [ ]:
# !pip install -qU langchain-chroma

In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:latest",
    temperature=0.5,
    max_tokens = 250
)

In [3]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="llama3.2:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)

def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


#### Output of the LLM Application

In [5]:
response = chain.invoke("What is MCP")

print(response)

MCP, or Model Context Protocol, is a standardization framework for building connected AI systems. It allows developers to create applications that can call APIs without custom integration code, much like how Custom GPTs use GPT Actions. MCP provides a way to determine which API call resolves the user's prompt, generate necessary JSON, and make API calls. This protocol offers similar capabilities across different AI models and vendors, enhancing flexibility and security in application development.


### Testing RAG Application with DeepEval
<img src="./img/RAGTesting.png" width="800" height="400" style="display: block; margin: auto;">

In [6]:
import deepeval

deepeval.login("confident_us_8k9P7QpyyKgjpa7yzXG0ULlki3JAwq0DPAstgNKA1x0=")

🎉🥳 Congratulations! You've successfully logged in! 🙌

In [7]:
!deepeval set-ollama llama3.2:latest

Settings updated for this session. To persist, use --save=dotenv[:path] 
(default .env.local) or set DEEPEVAL_DEFAULT_SAVE=dotenv:.env.local
🙌 Congratulations! You're now using a local Ollama model `llama3.2:latest` for
all evals that require an LLM.


In [8]:
test_data = [
    {
        "input": "What is MCP",
        "expected_output": "The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps."
    },
    {
        "input": "What is Relationship between function calling & Model Context Protocol",
        "expected_output": "The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes the development process by connecting AI applications to context while leveraging function calling to make API interactions more consistent across different applications and model vendors."
    },
    {
        "input": "What are the core components of MCP, just give the heading",
        "expected_output":""" 
                    - MCP Client
                    - MCP Servers
                    - Protocol Handshake
                    - Capability Discovery
                """
    }
]

### Creating Goldens

In [9]:
from deepeval.dataset import Golden, EvaluationDataset

goldens = []

for data in test_data:
    golden = Golden(
        input=data['input'],
        expected_output=data['expected_output']
    )
    
    goldens.append(golden)
    

dataset = EvaluationDataset(goldens=goldens)

In [11]:
dataset.push("TestGoldenDataSet")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=639060;https://app.confident-ai.com/project/cmisxih9601ubnp1fpqabp0df/datasets/cmiv9frhl01p6n11ftkn0x8x1\https://app.confident-ai.com/project/cmisxih9601ubnp1fpqabp0df/datasets/cmiv9frhl01p6n11ftkn0x8x1]8;;\

In [12]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='What is MCP', actual_output=None, expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='What is Relationship between function calling & Model Context Protocol', actual_output=None, expected_output='The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes th

In [13]:
dataset.pull(alias="TestGoldenDataSet")

c:\Users\Lenovo\Downloads\aiqaDemoSource\myenv\Lib\site-packages\rich\live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In [14]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(input='Who is the current president of the United States of America?', actual_output=None, expected_output='Joe Biden', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='Who introducted the GPT Model?', actual_output=None, expected_output='Open AI', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None), Golden(input='What is MCP', actual_output=None, expected_output='The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for c

In [15]:
from langchain_classic.chains import RetrievalQA

# It is going to use the LLM and Vector database stored information (RAG)
qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

In [16]:
response = qa_chain("What is MCP")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17292\3706383010.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = qa_chain("What is MCP")


In [17]:
# Is the data which is stored in Vector DB
retrieved_document = retrieve_and_format("What is MCP")
print(retrieved_document)

manual configuration with JSON files, highlighting an increasingly apparent gap between developer tooling and consumer use cases.Examples of MCP serversThe MCP ecosystem comprises a diverse range of servers including reference servers (created by the protocol maintainers as implementation examples), official integrations (maintained by companies for their platforms), and community servers (developed by independent contributors).Reference serversReference servers demonstrate core MCP

## Taxonomy
Learning center guide. Part of Descope's MCP series; foundational for security and deployment posts.

## Prerequisites / Related reading

**Related reading:**
- [Top 6 MCP Vulnerabilities](https://www.descope.com/blog/post/mcp-vulnerabilities) – Explores threats and mitigations
- [Diving Into the MCP Authorization Specification](https://www.descope.com/blog/post/mcp-auth-spec) – OAuth implementation discussion

# Authority

MCP standardization is foundational infrastructure for production AI ap

In [18]:
def query_with_context(question):
    retrieved_document = retrieve_and_format(question)
    response = qa_chain.run(question)
    return retrieved_document, response

In [19]:
actual, context = query_with_context("What is MCP")

actual, context

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17292\2296904857.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  response = qa_chain.run(question)


("manual configuration with JSON files, highlighting an increasingly apparent gap between developer tooling and consumer use cases.Examples of MCP serversThe MCP ecosystem comprises a diverse range of servers including reference servers (created by the protocol maintainers as implementation examples), official integrations (maintained by companies for their platforms), and community servers (developed by independent contributors).Reference serversReference servers demonstrate core MCP\n\n## Taxonomy\nLearning center guide. Part of Descope's MCP series; foundational for security and deployment posts.\n\n## Prerequisites / Related reading\n\n**Related reading:**\n- [Top 6 MCP Vulnerabilities](https://www.descope.com/blog/post/mcp-vulnerabilities) – Explores threats and mitigations\n- [Diving Into the MCP Authorization Specification](https://www.descope.com/blog/post/mcp-auth-spec) – OAuth implementation discussion\n\n# Authority\n\nMCP standardization is foundational infrastructure for p

- The function presumably takes a query (like a question) and returns two values:
    - actual → the direct answer or result to the query.
    - context → supporting information, background, or metadata that explains how/why the answer was derived.
- Tuple Unpacking: Python allows you to unpack multiple return values from a function into separate variables.
    - Example:
        - def sample():
            - return "Answer", "Extra info"
        - a, b = sample()
        - print(a)  # "Answer"
        - print(b)  # "Extra info"

### Creating LLMTestCase with Goldens

In [21]:
from deepeval.dataset import Golden
from deepeval.test_case import LLMTestCase
from typing import List


def convert_goldens_to_test_cases(goldens: List[Golden]) -> List[LLMTestCase]:
    test_cases = []
    for golden in goldens:
        context, rag_response = query_with_context(golden.input)
        test_case = LLMTestCase(
            input=golden.input,
            actual_output=rag_response,
            expected_output=golden.expected_output,
            retrieval_context=[context],
        )
        test_cases.append(test_case)
    return test_cases

data = convert_goldens_to_test_cases(dataset.goldens)
        

In [22]:
data

[LLMTestCase(input='Who is the current president of the United States of America?', actual_output="I don't know the answer to that specific question about the current president of the United States of America. My context doesn't provide this information, and I wouldn't want to give you an incorrect or outdated response. You can find the most accurate and up-to-date information by checking a reliable news source or the official government website.", expected_output='Joe Biden', context=None, retrieval_context=["## Descope's authority\nDescope secures production MCP deployments and advises enterprises on safe MCP integration.\n\nrequiredProductApp Use CasesAuthentication MethodsDevelopersResourcesCompanyLegalLeave a Descope reviewGithub Icon GreyLinkedin Icon GreyX Grey IconInstagram Grey LogoSlack Grey IconYoutube Grey IconBluesky SocialAll systems operationalCopyright © Descope Inc. All rights reserved.\n\n## Why this matters\n\neven managing branches. As an added security measure, Sup

In [ ]:
import deepeval.metrics
from deepeval.models import OllamaModel

ollama_model = OllamaModel(model="llama3.2:latest")
deepeval.evaluate(
    data, 
    metrics= [
        deepeval.metrics.AnswerRelevancyMetric(model = ollama_model)
        # deepeval.metrics.FaithfulnessMetric(),
        # deepeval.metrics.ContextualPrecisionMetric(),
        # deepeval.metrics.ContextualRelevancyMetric()
    ]
)
# The above four matrix can be executed all at once, to get the best result.
# But it need high computational resource. 

✨ You're running DeepEval's latest Answer Relevancy Metric! (using llama3.2:latest (Ollama), strict=False, 
async_mode=True)...

c:\Users\Lenovo\Downloads\aiqaDemoSource\myenv\Lib\site-packages\rich\live.py:256: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\asyncio\events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x0000020557211240> is already entered

Task was destroyed but it is pending!
task: <Task pending name='Task-133' coro=<_async_in_context.<locals>.run_in_context() done, defined at 
c:\Users\Lenovo\Downloads\aiqaDemoSource\myenv\Lib\site-packages\ipykernel\utils.py:57> wait_for=<Task pending 
name='Task-134' coro=<Kernel.shell_main() running at 
c:\Users\Lenovo\Downloads\aiqaDemoSource\myenv\Lib\site-packages\ipykernel\kernelbase.py:590> cb=[Task.__wakeup()]>
cb=[ZMQStream._run_callback.<locals>._log_error() at 
c:\Users\Lenovo\Downloads\aiqaDemoSource\myenv\Lib\site-packages\zmq\eventloop\zmqstream.py:563]>

c:\Users\Lenovo\Downloads\aiqaDemoSource\myenv\Lib\site-packages\pydantic\_internal\_config.py:155: RuntimeWarning:
coroutine 'Kernel.shell_main' was never awaited
  return self.config_dict[name]
RuntimeWarning: Enable tracemalloc to get the object allocation traceback

Task was destroyed but it is pending!
task: <Task pending name='Task-134' coro=<Kernel.shell_main() running at 
c:\Users\Lenovo\Downloads\aiqaDemoSource\myenv\Lib\site-packages\ipykernel\kernelbase.py:590> cb=[Task.__wakeup()]>

RetryError: RetryError[<Future at 0x2056ad71e90 state=finished raised TimeoutError>]

: 